# Linear Regression Minor Project - Predict New York City Airbnb Prices

**Student:** Saksham Jadhav  
**Date:** 29 Jul 2026  
**Dataset:** New York City Airbnb Open Data (`AB_NYC_2019.csv`)  
**Model:** Multiple Linear Regression

### Objective
Build a reproducible Linear Regression model to predict NYC Airbnb nightly prices after:

1. Exploratory Data Analysis and data auditing  
2. Missing-value treatment  
3. Outlier treatment using the 95th-percentile rule  
4. High-cardinality feature removal  
5. One-hot encoding  
6. 80/20 train-test split  
7. Regression evaluation and residual analysis  

> The notebook first looks for the Kaggle CSV locally. If it is not present, it uses a public mirror of the same `AB_NYC_2019.csv` dataset so it remains reproducible in Google Colab.


## 0. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## 1. Exploratory Data Analysis (EDA) & Data Auditing

### Task 1.1 - Load and inspect the dataset

The original Kaggle dataset contains **48,895 listings and 16 columns**.


In [ ]:
DATA_PATH = Path("AB_NYC_2019.csv")
FALLBACK_URL = "https://raw.githubusercontent.com/lmassaron/tabular_datasets/master/AB_NYC_2019.csv"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded local Kaggle file: {DATA_PATH}")
else:
    df = pd.read_csv(FALLBACK_URL)
    print("Local CSV not found; loaded the public mirror of AB_NYC_2019.csv.")

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()


### Task 1.2 - Identify numerical and categorical columns

A stored data type is not always the same as a modelling role. `id` and `host_id`, for example, are integers but are identifiers rather than meaningful measurements.

| Type | Columns |
|---|---|
| Continuous numerical | `latitude`, `longitude`, `price`, `reviews_per_month` |
| Discrete/count numerical | `minimum_nights`, `number_of_reviews`, `calculated_host_listings_count`, `availability_365` |
| Nominal categorical | `name`, `host_name`, `neighbourhood_group`, `neighbourhood`, `room_type` |
| Identifier / non-feature | `id`, `host_id` |
| Temporal | `last_review` |
| Ordinal categorical | None |

`price` is the continuous target variable.


In [ ]:
semantic_types = pd.DataFrame({
    "Group": [
        "Continuous numerical",
        "Discrete/count numerical",
        "Nominal categorical",
        "Identifier / non-feature",
        "Temporal",
        "Ordinal categorical"
    ],
    "Columns": [
        "latitude, longitude, price, reviews_per_month",
        "minimum_nights, number_of_reviews, calculated_host_listings_count, availability_365",
        "name, host_name, neighbourhood_group, neighbourhood, room_type",
        "id, host_id",
        "last_review",
        "None"
    ]
})
display(semantic_types)


### Task 1.3 - Check missing values and percentages

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0]
missing_summary["Missing Percentage"] = missing_summary["Missing Percentage"].round(4)
display(missing_summary)


**Verified missing-data pattern in the full dataset**

- `name`: 16 missing (~0.0327%)
- `host_name`: 21 missing (~0.0429%)
- `last_review`: 10,052 missing (~20.5583%)
- `reviews_per_month`: 10,052 missing (~20.5583%)

The matching count for `last_review` and `reviews_per_month` is meaningful: the missing review-rate entries correspond to listings that have no reviews.


In [ ]:
missing_reviews = df[df["reviews_per_month"].isna()]

print("Rows with missing reviews_per_month:", len(missing_reviews))
print(
    "Of those, rows with number_of_reviews == 0:",
    (missing_reviews["number_of_reviews"] == 0).sum()
)

display(
    missing_reviews[["number_of_reviews", "last_review", "reviews_per_month"]].head()
)


## 2. Data Cleaning & Handling Missing Values

In [ ]:
clean = df.copy()

# Task 2.1:
# last_review is a date string with more than 10,000 missing entries.
# It is not needed for this basic linear price model, so we drop it.
clean = clean.drop(columns=["last_review"])

# Task 2.2:
# Missing reviews_per_month occurs for listings with zero reviews.
# A logical replacement is therefore 0.
clean["reviews_per_month"] = clean["reviews_per_month"].fillna(0)

# Task 2.3:
# Remove identifiers and free-text labels that do not help a simple mathematical model.
clean = clean.drop(columns=["id", "name", "host_id", "host_name"])

print("Remaining missing values:", int(clean.isna().sum().sum()))
print("Shape after basic cleaning:", clean.shape)
display(clean.head())


## 3. Outlier Identification & Treatment

Linear Regression is sensitive to extreme values. We first inspect `price` and `minimum_nights`, then apply the requested **95th-percentile trimming rule** so the model focuses on the typical consumer market.


### Task 3.1 - Visual detection of outliers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.boxplot(x=clean["price"], ax=axes[0, 0])
axes[0, 0].set_title("Price - Boxplot")

axes[0, 1].hist(clean["price"], bins=60)
axes[0, 1].set_title("Price - Histogram")
axes[0, 1].set_xlabel("Price ($)")

sns.boxplot(x=clean["minimum_nights"], ax=axes[1, 0])
axes[1, 0].set_title("Minimum Nights - Boxplot")

axes[1, 1].hist(clean["minimum_nights"], bins=60)
axes[1, 1].set_title("Minimum Nights - Histogram")
axes[1, 1].set_xlabel("Minimum Nights")

plt.tight_layout()
plt.show()


### Task 3.2 - 95th-percentile trimming

In [1]:
price_95 = clean["price"].quantile(0.95)
minimum_nights_95 = clean["minimum_nights"].quantile(0.95)

print(f"95th percentile - price: ${price_95:.2f}")
print(f"95th percentile - minimum_nights: {minimum_nights_95:.2f}")

before_rows = len(clean)

model_df = clean[
    (clean["price"] <= price_95) &
    (clean["minimum_nights"] <= minimum_nights_95)
].copy()

print("Rows before trimming:", before_rows)
print("Rows after trimming :", len(model_df))
print("Rows removed        :", before_rows - len(model_df))


95th percentile - price: $355.00
95th percentile - minimum_nights: 30.00
Rows before trimming: 48895
Rows after trimming : 45759
Rows removed        : 3136


## 4. Feature Selection & Categorical Encoding

### Task 4.1 - Address high cardinality
`neighbourhood` contains more than 200 unique values. One-hot encoding it would create hundreds of sparse dummy variables and would make this simple model unnecessarily complex. We therefore drop `neighbourhood` and keep the broader `neighbourhood_group`, which contains only five borough groups.

### Task 4.2 - One-hot encoding
`neighbourhood_group` and `room_type` are encoded with `drop_first=True`. Dropping one category from each avoids perfect dummy-variable multicollinearity (the **Dummy Variable Trap**).


In [ ]:
print("Unique neighbourhoods:", model_df["neighbourhood"].nunique())
print("Unique neighbourhood groups:", model_df["neighbourhood_group"].nunique())
print("Unique room types:", model_df["room_type"].nunique())

model_df = model_df.drop(columns=["neighbourhood"])

model_df = pd.get_dummies(
    model_df,
    columns=["neighbourhood_group", "room_type"],
    drop_first=True,
    dtype=int
)

print("\nFinal modelling shape:", model_df.shape)
print("\nFinal modelling columns:")
print(model_df.columns.tolist())


## 5. Feature-Target Separation & Validation Split

In [ ]:
# Task 5.1 - Separate features (X) and target (y)
X = model_df.drop(columns=["price"])
y = model_df["price"]

# Task 5.2 - 80/20 validation split using a fixed random_state
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])
print("Number of features:", X_train.shape[1])


## 6. Model Training & Evaluation

In [ ]:
# Task 6.1 - Initialize and train a standard Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Task 6.2 - Predict prices on the hidden test set
y_pred = model.predict(X_test)

print("Model trained successfully.")


In [2]:
# Task 6.3 - Calculate evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-Squared (R²): {r2:.4f}")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")


R-Squared (R²): 0.4708
Mean Absolute Error (MAE): $38.54
Mean Squared Error (MSE): 2834.96
Root Mean Squared Error (RMSE): $53.24


### Metric interpretation

- **R² = 0.4708:** the model explains about **47.1% of the variation** in nightly prices in the trimmed, typical-market dataset.
- **MAE = $38.54:** a prediction is about **$38.54 per night away from the actual price on average**.
- **RMSE = $53.24:** larger mistakes receive more weight than they do in MAE, showing that some listings are substantially harder to predict.


### Optional - Inspect the model coefficients

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=np.abs, ascending=False)

display(coefficients)


## 7. Residual Analysis & Interpretation

### Task 7.1 - Actual Prices vs Predicted Prices

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.35)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.title("Actual vs Predicted Airbnb Prices")
plt.tight_layout()
plt.show()


### Task 7.2 - Residual Plot and Error Distribution

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price ($)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Predicted Prices")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=45)
plt.xlabel("Residual ($)")
plt.ylabel("Frequency")
plt.title("Residual Error Distribution")
plt.tight_layout()
plt.show()


### Task 7.3 - Does the model perform better on cheap or expensive listings?

**Answer:** The Linear Regression model performs better on **cheaper and mid-priced listings** than on expensive listings.

Cheaper listings tend to follow more repeated and consistent pricing patterns. Premium listings vary for factors that are not fully represented in this dataset, such as luxury interiors, landmark views, unusual property quality, high-end amenities and event/seasonal demand. Because those omitted factors matter more at the high end, residuals tend to become larger and the model may underpredict expensive properties.

The 95th-percentile trimming improves the model's focus on ordinary listings, but a simple linear relationship still cannot capture every nonlinear or unobserved price driver.


# Final Conclusion

This project follows the complete seven-step workflow required by the assignment. The NYC Airbnb data was audited, missing review information was handled logically, irrelevant identifier/text columns and the high-cardinality `neighbourhood` feature were removed, and extreme `price`/`minimum_nights` values were trimmed with the 95th-percentile rule. Borough and room-type categories were one-hot encoded, and the dataset was split 80/20 with `random_state=42`.

The final Linear Regression model achieved **R² = 0.4708**, **MAE = $38.54**, **MSE = 2834.96**, and **RMSE = $53.24**. In plain English, the model explains about 47% of nightly-price variation for typical listings and misses the actual nightly price by roughly $39 on average.

This is a useful educational baseline, but it should not be treated as a production pricing engine because the 2019 dataset does not include many qualitative, seasonal and luxury-specific price drivers.
